In [1]:
import os 
import langchain
from pydantic import BaseModel,Field
from typing import Dict, Optional
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate , PromptTemplate


In [2]:
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser,JsonOutputParser
import re
import json

def extract_response(text: str) -> str:
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return json.loads(cleaned)
class iskillQuery(BaseModel):
    result: bool = Field(..., description="The user query if query is  asking for skills or not")

def is_skill_query_prompt(parser=None):
    template = """Given a user query, determine if the user is asking about the skills available.
                query: {query_for_skill}
                Return True or False only. in JSON format"""
    
    return PromptTemplate(input_variables=["query_for_skill"], template=template,)# partial_variables={"format_instructions": parser.get_format_instructions()})

structred_response = StrOutputParser(pydantic_object=iskillQuery)


In [3]:
user_query = "I want to read a image file  ?"  
os.environ["GROQ_API_KEY"] = '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'
llm = ChatGroq(model="qwen/qwen3-32b")

chain = is_skill_query_prompt() | llm | structred_response


In [4]:
llm.model_name

'qwen/qwen3-32b'

In [74]:
score_data = chain.invoke({
        'query_for_skill': user_query
        }
    )
response = extract_response(score_data)
print(response['result'])

False
